<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright 2026 Evan Gress -->

# 03 · Beam design: units, sizing, and optimization

Analysis asks *how much does this beam deflect?* Design asks *what beam deflects no more than 5 mm?* This notebook does both.

## 1. Units with Pint
`Q_(value, unit)` makes a quantity. Arithmetic carries the units and `.to()` converts.

In [ ]:
from fpc import Beam, Q_

beam = Beam(length=Q_(2.0, "m"), width=Q_(50, "mm"), height=Q_(100, "mm"))
beam.max_deflection_point_load(load=Q_(5, "kN")).to("mm")

Mixing up units fails loudly instead of giving a wrong number:

In [ ]:
try:
    Beam(length=Q_(2.0, "s"), width=Q_(50, "mm"), height=Q_(100, "mm"))
except ValueError as error:
    print("Caught:", error)

## 2. Sizing by root finding
`Design.rectangular_beam_height_for_deflection` solves *deflection(h) − limit = 0* with `brentq`. Plain SI floats here.

In [ ]:
from fpc import Design

design = Design()
h = design.rectangular_beam_height_for_deflection(
    load=5000, length=2.0, modulus=200e9, width=0.05, max_deflection=0.001,
)
f"required height = {h * 1000:.1f} mm"

## 3. Optimization with constraints
Minimize mass subject to stress, deflection, and a height ≤ 4 × width rule. The constraints that end up at their limits are the ones that *drive* the design.

In [ ]:
design.lightest_rectangular_beam(
    load=5000, length=2.0, modulus=200e9, density=7850,
    allowable_stress=150e6, max_deflection=0.005,
)

Sweep the load to see when the governing requirement switches from deflection to stress:

In [ ]:
import pandas as pd

rows = []
for load in (2e3, 5e3, 10e3, 20e3, 40e3):
    r = design.lightest_rectangular_beam(load=load, length=2.0, modulus=200e9, density=7850,
                                         allowable_stress=150e6, max_deflection=0.01)
    rows.append({"load_kN": load / 1e3, "mass_kg": round(r["mass"], 1),
                 "governs": ", ".join(r["active"])})
pd.DataFrame(rows)

## 4. Catalogue selection
Real beams come from a catalogue. `Properties.lightest_w_shape` filters and sorts the AISC table with pandas; Pint converts the required section modulus to the catalogue's units.

In [ ]:
from fpc import Properties

moment = Q_(40, "kN") * Q_(6, "m") / 4
required_sx = (moment / (0.66 * Q_(345, "MPa"))).to("in**3")
print(f"required Sx = {required_sx:.1f~P}")
Properties().lightest_w_shape(required_sx_in3=required_sx.magnitude)

See `examples/beam_selection.py` for the full version with a deflection check.